In [6]:
import pandas as pd
import json

# Load the dataset
file_path = '../diff_case_claims_with_negatives.csv'
df = pd.read_csv(file_path)

print(f"Dataset loaded. Shape: {df.shape}")
display(df.head())

Dataset loaded. Shape: (11818, 10)


,claim,case_name,judgement,explanation,raw_output,type,overruling_case,fact_id,label,is_negation
0,The Constitution does not recognize a right to...,"[""Roe v. Wade""]",consistent,The case evidence explicitly states that the C...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,0,0,True
1,The Constitution protects a person's right to ...,"[""Roe v. Wade""]",consistent,The case evidence explicitly states that the C...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,0,1,False
2,States are permitted to impose regulations on ...,"[""Roe v. Wade""]",consistent,"The case evidence explicitly states that ""In t...","```json\n{\n ""explanation"": ""The case evide...",none,NaN,1,0,True
3,States cannot restrict abortion during the fir...,"[""Roe v. Wade""]",consistent,"The case evidence explicitly states that ""In t...","```json\n{\n ""explanation"": ""The case evide...",none,NaN,1,1,False
4,"During the second three months of pregnancy, s...","[""Roe v. Wade""]",consistent,The case evidence states that in the second tr...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,2,1,False


In [ ]:
# Summary Statistics

print("--- General Statistics ---")
print(f"Total Samples: {len(df)}")
print(f"Unique Fact IDs: {df['fact_id'].nunique()}")
print(f"Unique Cases: {df['case_name'].nunique()}")

print("\n--- Class Distribution ---")
print(df['class'].value_counts())
print(df['class'].value_counts(normalize=True))

print("\n--- Label Distribution ---")
print(df['label'].value_counts())

print("\n--- Type Distribution ---")
print(df['type'].value_counts())

--- General Statistics ---
Total Samples: 11818
Unique Fact IDs: 5909
Unique Cases: 3214

--- Label Distribution ---
label
0    5909
1    5909
Name: count, dtype: int64
label
0    0.5
1    0.5
Name: proportion, dtype: float64

--- Type Distribution (Original Claims) ---
type
none          5346
overruling     343
confirmed      220
Name: count, dtype: int64
type
none          0.904722
overruling    0.058047
confirmed     0.037231
Name: proportion, dtype: float64

--- Negation Distribution ---
is_negation
True     5909
False    5909
Name: count, dtype: int64


In [ ]:
# Examples

def print_example(fact_id):
    rows = df[df['fact_id'] == fact_id]
    print(f"--- Fact ID: {fact_id} ---")
    # Print shared metadata from first row
    first_row = rows.iloc[0]
    print(f"Case: {first_row['case_name']}")
    print(f"Facts: {first_row['facts'][:200]}..." if isinstance(first_row['facts'], str) else f"Facts: {first_row['facts']}")
    print(f"Question: {first_row['question']}")
    print(f"Conclusion: {first_row['conclusion'][:200]}..." if isinstance(first_row['conclusion'], str) else f"Conclusion: {first_row['conclusion']}")
    print("-" * 20)
    
    for _, row in rows.iterrows():
        print(f"Class: {row['class']} (Label {row['label']})")
        print(f"Type: {row['type']}")
        print(f"Claim: {row['claim']}")
        print("-" * 10)
    print("=" * 30)

print("--- Random Example ---")
random_id = df['fact_id'].sample(1).iloc[0]
print_example(random_id)

print("\n--- Overruling Example ---")
overruling_ids = df[df['class'] == 'Overruled']['fact_id'].unique()
if len(overruling_ids) > 0:
    print_example(overruling_ids[0])

print("\n--- Supported Example ---")
supported_ids = df[df['class'] == 'Supported']['fact_id'].unique()
if len(supported_ids) > 0:
    print_example(supported_ids[0])

--- Random Example Pair ---
--- Fact ID: 784 (Type: none) ---
Case: ["North Haven Bd. of Educ. v. Bell"]
Original Claim (Label 1): Government regulations that enforce anti-discrimination laws stay in effect if Congress does not pass a resolution to block them.
Negation (Label 0): Government regulations that enforce anti-discrimination laws are automatically invalidated if a resolution to block them is introduced in Congress, even if that resolution does not pass.
------------------------------

--- Overruling Example ---
--- Fact ID: 12 (Type: overruling) ---
Case: ["Kahn v. Shevin", "United States v. Virginia"]
Original Claim (Label 1): Gender-based rules in government programs must be fair and not based on stereotypes to meet constitutional standards.
Negation (Label 0): Gender-based rules in government programs can be constitutional as long as a separate, though not necessarily identical, program is created for the excluded gender, regardless of differences in prestige, resources, o

In [5]:
import numpy as np

# 1. Get unique fact_ids
unique_fact_ids = df['fact_id'].unique()

# 2. Sample 1000 unique fact_ids
n_sample = 1000
if len(unique_fact_ids) < n_sample:
    print(f"Warning: Only {len(unique_fact_ids)} unique fact_ids available. Sampling all.")
    n_sample = len(unique_fact_ids)

sampled_ids = np.random.choice(unique_fact_ids, n_sample, replace=False)

# 3. Select one claim per fact_id (randomly positive or negative)
# Filter for the sampled IDs, then group by fact_id and sample 1
subsample = df[df['fact_id'].isin(sampled_ids)].groupby('fact_id').sample(n=1)

# 4. Randomize order
subsample = subsample.sample(frac=1).reset_index(drop=True)

# 5. Statistics
print(f"--- Subsample Statistics (n={len(subsample)}) ---")

print("\nLabel Distribution (Positive vs Negative):")
print(subsample['label'].value_counts())
print(subsample['label'].value_counts(normalize=True))

print("\nType Distribution:")
print(subsample['type'].value_counts())
print(subsample['type'].value_counts(normalize=True))

# Verify uniqueness
print(f"\nAre all fact_ids unique? {subsample['fact_id'].is_unique}")

# Display head
display(subsample.head())

--- Subsample Statistics (n=1000) ---

Label Distribution (Positive vs Negative):
label
0    509
1    491
Name: count, dtype: int64
label
0    0.509
1    0.491
Name: proportion, dtype: float64

Type Distribution:
type
none          903
overruling     62
confirmed      35
Name: count, dtype: int64
type
none          0.903
overruling    0.062
confirmed     0.035
Name: proportion, dtype: float64

Are all fact_ids unique? True


,claim,case_name,judgement,explanation,raw_output,type,overruling_case,fact_id,label,is_negation
0,The Patent Trial and Appeal Board must decide ...,"[""SAS Institute Inc. v. Iancu""]",consistent,The case evidence explicitly states that the S...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,5448,1,False
1,Congress cannot regulate manufacturing or prod...,"[""Hammer v. Dagenhart""]",consistent,The case evidence states that the Court found ...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,5162,1,False
2,Government regulations requiring personal info...,"[""Whalen v. Roe""]",consistent,The case evidence states the Court found the s...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,401,1,False
3,If the parties aren't from different states or...,"[""Caterpillar Inc. v. Lewis"", ""Grupo Dataflux ...",consistent,The claim states that a case must be dismissed...,"```json\n{\n ""explanation"": ""The claim stat...",overruling,"Grupo Dataflux v. Atlas Global Group, L.P.",2901,1,False
4,The government can legally count all household...,"[""Bowen v. Gilliard""]",consistent,The case evidence details the Deficit Reductio...,"```json\n{\n ""explanation"": ""The case evide...",none,NaN,1123,1,False


In [7]:
# Process Data
def process_data(df, exclude_overruling=False):
    # Print first example of case_name
    print(f"First case_name raw: {df['case_name'].iloc[0]}")
    
    # Parse case_name (it's a JSON list string)
    df['clean_case_name'] = df['case_name'].apply(lambda x: json.loads(x)[0] if isinstance(x, str) and x.startswith('[') else x)
    
    # Load metadata
    meta_path = '../clean_data_with_details.csv'
    meta_df = pd.read_csv(meta_path)
    
    # Merge
    # We need facts, api_question, api_conclusion from metadata
    # Match clean_case_name with name
    merged = df.merge(meta_df[['name', 'facts', 'api_question', 'api_conclusion']], 
                      left_on='clean_case_name', right_on='name', how='left')
    
    # Rename columns
    merged = merged.rename(columns={'api_question': 'question', 'api_conclusion': 'conclusion'})
    
    # Create class column
    def get_class(row):
        if row['type'] == 'overruling':
            return 'Overruled'
        elif row['is_negation']:
            return 'Refuted'
        else:
            return 'Supported'
            
    merged['class'] = merged.apply(get_class, axis=1)
    
    # Update label to match class (0: Refuted, 1: Supported, 2: Overruled)
    label_map = {'Refuted': 0, 'Supported': 1, 'Overruled': 2}
    merged['label'] = merged['class'].map(label_map)
    
    # Filter overruling if requested
    if exclude_overruling:
        merged = merged[merged['class'] != 'Overruled']
        
    # Drop columns
    cols_to_drop = ['judgement', 'explanation', 'raw_output', 'is_negation', 'clean_case_name', 'name']
    merged = merged.drop(columns=cols_to_drop, errors='ignore')
    
    return merged

# Apply processing
df = process_data(df, exclude_overruling=True)

print(f"Processed dataset shape: {df.shape}")
display(df.head())
print("\nClass Distribution:")
print(df['class'].value_counts())

First case_name raw: ["Roe v. Wade"]
Processed dataset shape: (11374, 10)


,claim,case_name,type,overruling_case,fact_id,label,facts,question,conclusion,class
0,The Constitution does not recognize a right to...,"[""Roe v. Wade""]",none,NaN,0,0,"In 1970, Jane Roe (a fictional name used in co...",Does the Constitution recognize a woman's righ...,Inherent in the Due Process Clause of the Four...,Refuted
1,The Constitution protects a person's right to ...,"[""Roe v. Wade""]",none,NaN,0,1,"In 1970, Jane Roe (a fictional name used in co...",Does the Constitution recognize a woman's righ...,Inherent in the Due Process Clause of the Four...,Supported
2,States are permitted to impose regulations on ...,"[""Roe v. Wade""]",none,NaN,1,0,"In 1970, Jane Roe (a fictional name used in co...",Does the Constitution recognize a woman's righ...,Inherent in the Due Process Clause of the Four...,Refuted
3,States cannot restrict abortion during the fir...,"[""Roe v. Wade""]",none,NaN,1,1,"In 1970, Jane Roe (a fictional name used in co...",Does the Constitution recognize a woman's righ...,Inherent in the Due Process Clause of the Four...,Supported
4,"During the second three months of pregnancy, s...","[""Roe v. Wade""]",none,NaN,2,1,"In 1970, Jane Roe (a fictional name used in co...",Does the Constitution recognize a woman's righ...,Inherent in the Due Process Clause of the Four...,Supported



Class Distribution:
class
Refuted      5687
Supported    5687
Name: count, dtype: int64


In [8]:
# Sample 500 unique fact_ids
n_sample = 500
unique_fact_ids = df['fact_id'].unique()

if len(unique_fact_ids) < n_sample:
    print(f"Warning: Only {len(unique_fact_ids)} unique fact_ids available. Sampling all.")
    n_sample = len(unique_fact_ids)

sampled_ids = np.random.choice(unique_fact_ids, n_sample, replace=False)

# Select one claim per fact_id
sampled_df = df[df['fact_id'].isin(sampled_ids)].groupby('fact_id').sample(n=1)

# Randomize order
sampled_df = sampled_df.sample(frac=1).reset_index(drop=True)

print(f"Sampled shape: {sampled_df.shape}")
print(f"Unique fact_ids: {sampled_df['fact_id'].nunique()}")

# Save to CSV
output_path = 'sampled_claims_500_jacob.csv'
sampled_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

display(sampled_df.head())

Sampled shape: (500, 10)
Unique fact_ids: 500
Saved to sampled_claims_500_jacob.csv


,claim,case_name,type,overruling_case,fact_id,label,facts,question,conclusion,class
0,The military can require uniforms that do not ...,"[""Goldman v. Weinberger""]",none,NaN,1035,1,Goldman was a commissioned officer in the Unit...,Did the Air Force Regulation violate the Free ...,The Court held that the Air Force regulation d...,Supported
1,The right to travel internationally is absolut...,"[""Haig v. Agee""]",none,NaN,698,0,"In 1974, Philip Agee, a former employee of the...","Did the President, acting through the Secretar...","In a 7-to-2 decision, the Court held that Pass...",Refuted
2,"To start a lawsuit, you must show that the cou...","[""Steel Company v. Citizens for Better Environ...",none,NaN,2076,1,"In 1995, Citizens For A Better Environment, a ...",Does an environmental organization have standi...,No and the Court did not answer the question. ...,Supported
3,Public school teachers can be forbidden from t...,"[""Epperson v. Arkansas""]",none,NaN,4869,0,The Arkansas legislature passed a law prohibit...,Does a law forbidding the teaching of evolutio...,Yes. Seven members of the Court held that the ...,Refuted
4,Federal courts *do* assume an immigrant's test...,"[""Garland v. Dai""]",none,NaN,5846,0,"Ming Dai, a native and citizen of China, sough...",Can a court of appeals presume that an immigra...,A court of appeals cannot presume that an immi...,Refuted
